In [3]:
from typing import TypedDict,Annotated,Literal
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel,Field
from langchain_core.messages import SystemMessage, HumanMessage,BaseMessage
import operator
from langgraph.checkpoint.memory import InMemorySaver
load_dotenv()

# 1. Initialize the model
model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.7,
    max_tokens=500,
)

llm=model


In [4]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza go to therapy?\n\nBecause it had too many “cheese”y feelings and was feeling a bit *crusty* about its identity! 🍕😄',
 'explanation': '**Why the joke works – a quick breakdown**\n\n| Element | What it is | Why it’s funny |\n|---------|------------|----------------|\n| **Pizza “going to therapy”** | A pizza is an inanimate food item, yet the joke treats it like a human who can have emotional problems. | The absurdity of a pizza seeking help is the initial punchline hook—nothing else in everyday life would “go to therapy.” |\n| **Too many “cheese”y feelings** | “Cheese” is literally what’s on a pizza, and “cheesy” (pronounced the same) is slang for something that’s corny, over‑the‑top, or inauthentic. | The double meaning creates a pun: the pizza’s emotions are literally *cheese*‑laden, but at the same time it’s “cheesy” in a figurative sense. |\n| **Feeling a bit *crusty* about its identity** | “Crusty” can mean having a crust (pizza’s outer

In [5]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to therapy?\n\nBecause it had too many “cheese”y feelings and was feeling a bit *crusty* about its identity! 🍕😄', 'explanation': '**Why the joke works – a quick breakdown**\n\n| Element | What it is | Why it’s funny |\n|---------|------------|----------------|\n| **Pizza “going to therapy”** | A pizza is an inanimate food item, yet the joke treats it like a human who can have emotional problems. | The absurdity of a pizza seeking help is the initial punchline hook—nothing else in everyday life would “go to therapy.” |\n| **Too many “cheese”y feelings** | “Cheese” is literally what’s on a pizza, and “cheesy” (pronounced the same) is slang for something that’s corny, over‑the‑top, or inauthentic. | The double meaning creates a pun: the pizza’s emotions are literally *cheese*‑laden, but at the same time it’s “cheesy” in a figurative sense. |\n| **Feeling a bit *crusty* about its identity** | “Crusty” can mean having a c

In [6]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to therapy?\n\nBecause it had too many “cheese”y feelings and was feeling a bit *crusty* about its identity! 🍕😄', 'explanation': '**Why the joke works – a quick breakdown**\n\n| Element | What it is | Why it’s funny |\n|---------|------------|----------------|\n| **Pizza “going to therapy”** | A pizza is an inanimate food item, yet the joke treats it like a human who can have emotional problems. | The absurdity of a pizza seeking help is the initial punchline hook—nothing else in everyday life would “go to therapy.” |\n| **Too many “cheese”y feelings** | “Cheese” is literally what’s on a pizza, and “cheesy” (pronounced the same) is slang for something that’s corny, over‑the‑top, or inauthentic. | The double meaning creates a pun: the pizza’s emotions are literally *cheese*‑laden, but at the same time it’s “cheesy” in a figurative sense. |\n| **Feeling a bit *crusty* about its identity** | “Crusty” can mean having a 

In [7]:
config2={"configurable":{"thread_id":"2"}}
workflow.invoke({'topic':'pasta'},config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti start a band?  \nBecause it already had the perfect *pasta*-t rhythm! 🍝🎸',
 'explanation': ''}